In [13]:
import os, sys, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import soundfile as sf
import librosa

sys.path.append("..")          # so we can import ser_pipeline.py from the repo root
import ser_pipeline as sp

os.makedirs("../images", exist_ok=True)
os.makedirs("../data", exist_ok=True)
sns.set_theme(style="whitegrid")

# 1. Load the metadata

In [14]:
df = sp.build_metadata("../data/RAVDESS", "speech")
print(df.shape)          # expect (1440, 11)
df.head()

(1440, 11)


,path,filename,modality,vocal_channel,emotion_id,emotion,intensity,statement,repetition,actor,gender
0,..\data\RAVDESS\Actor_01\03-01-01-01-01-01-01.wav,03-01-01-01-01-01-01.wav,audio-only,speech,1,neutral,normal,Kids are talking by the door,1,1,male
1,..\data\RAVDESS\Actor_01\03-01-01-01-01-02-01.wav,03-01-01-01-01-02-01.wav,audio-only,speech,1,neutral,normal,Kids are talking by the door,2,1,male
2,..\data\RAVDESS\Actor_01\03-01-01-01-02-01-01.wav,03-01-01-01-02-01-01.wav,audio-only,speech,1,neutral,normal,Dogs are sitting by the door,1,1,male
3,..\data\RAVDESS\Actor_01\03-01-01-01-02-02-01.wav,03-01-01-01-02-02-01.wav,audio-only,speech,1,neutral,normal,Dogs are sitting by the door,2,1,male
4,..\data\RAVDESS\Actor_01\03-01-02-01-01-01-01.wav,03-01-02-01-01-01-01.wav,audio-only,speech,2,calm,normal,Kids are talking by the door,1,1,male


# 2. Inventory and missing values


In [15]:
print("Missing values per column:\n", df.isna().sum(), "\n")
print("Clips:", len(df), "| actors:", df["actor"].nunique())
print("Clips per actor:", df.groupby("actor").size().unique())
print("\nEmotion counts:\n", df["emotion"].value_counts())
print("\nGender counts:\n", df["gender"].value_counts())

Missing values per column:
 path             0
filename         0
modality         0
vocal_channel    0
emotion_id       0
emotion          0
intensity        0
statement        0
repetition       0
actor            0
gender           0
dtype: int64 

Clips: 1440 | actors: 24
Clips per actor: [60]

Emotion counts:
 emotion
calm         192
happy        192
sad          192
angry        192
fearful      192
disgust      192
surprised    192
neutral       96
Name: count, dtype: int64

Gender counts:
 gender
male      720
female    720
Name: count, dtype: int64


**Result:** 1,440 clips, 24 actors, 60 clips per actor, no missing values. Neutral has 96 clips and every other emotion has 192 (class imbalance to handle in modeling). Gender is balanced (720 male, 720 female).

# 3. Raw format audit


In [16]:
rows = []
for p in df["path"]:
    try:
        i = sf.info(p)
        rows.append({"path": p, "sr": i.samplerate, "channels": i.channels,
                     "subtype": i.subtype, "duration": i.duration, "readable": True})
    except Exception as e:
        rows.append({"path": p, "sr": None, "channels": None,
                     "subtype": None, "duration": None, "readable": False})
fmt = pd.DataFrame(rows)
print(fmt[["sr", "channels", "subtype"]].value_counts(dropna=False))
print("Unreadable files:", (~fmt["readable"]).sum())

sr     channels  subtype
48000  1         PCM_16     1435
       2         PCM_16        5
Name: count, dtype: int64
Unreadable files: 0


In [17]:
stereo = fmt[fmt["channels"] == 2].copy()
stereo["filename"] = stereo["path"].apply(os.path.basename)
print("Stereo files:", len(stereo))
stereo[["filename", "sr", "channels", "duration"]]

Stereo files: 5


,filename,sr,channels,duration
5,03-01-02-01-01-02-01.wav,48000,2,3.603583
55,03-01-08-01-02-02-01.wav,48000,2,3.269917
247,03-01-02-01-02-02-05.wav,48000,2,4.237563
1154,03-01-03-01-02-01-20.wav,48000,2,3.470125
1177,03-01-06-01-01-02-20.wav,48000,2,4.371021


**Result:** all files are 48 kHz, 16-bit PCM and readable. 1,435 files are mono and **5 are stereo**. The stereo files are not removed: loading with `mono=True` averages the two channels, so every clip becomes mono. All audio is also resampled to 22.05 kHz later.

# 4. Duplicates
duplicated clips can end up in both train and test sets and inflate scores. We check both duplicated filenames and identical audio content (MD5 hash of the file).

In [18]:
def md5(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

df["md5"] = [md5(p) for p in df["path"]]
print("Duplicate filenames:", df["filename"].duplicated().sum())
print("Duplicate audio files:", df["md5"].duplicated().sum())

Duplicate filenames: 0
Duplicate audio files: 1


In [19]:
dups = df[df["md5"].duplicated(keep=False)].sort_values("md5")
dups[["filename", "emotion", "intensity", "actor", "repetition", "md5"]]

,filename,emotion,intensity,actor,repetition,md5
374,03-01-03-01-02-01-07.wav,happy,normal,7,1,945f27c7a9cc3cc3b2a836db2651eb11
375,03-01-03-01-02-02-07.wav,happy,normal,7,2,945f27c7a9cc3cc3b2a836db2651eb11


# 5. Per-clip statistics


In [20]:
stats = []
for p in df["path"]:
    y, sr = librosa.load(p, sr=22050, mono=True)
    yt, _ = librosa.effects.trim(y, top_db=25)
    stats.append({
        "peak": np.abs(y).max(),
        "rms": np.sqrt(np.mean(y ** 2)),
        "clip_frac": np.mean(np.abs(y) >= 0.999),
        "dur_raw": len(y) / sr,
        "dur_trim": len(yt) / sr,
        "silence_ratio": 1 - len(yt) / len(y),
    })
df = pd.concat([df.reset_index(drop=True), pd.DataFrame(stats)], axis=1)
df[["peak", "rms", "clip_frac", "dur_raw", "dur_trim", "silence_ratio"]].describe().round(3)

,peak,rms,clip_frac,dur_raw,dur_trim,silence_ratio
count,1440.000,1440.000,1440.0,1440.000,1440.000,1440.000
mean,0.176,0.018,0.0,3.701,1.783,0.522
std,0.212,0.022,0.0,0.337,0.381,0.071
min,0.005,0.001,0.0,2.936,1.045,0.123
25%,0.048,0.005,0.0,3.470,1.533,0.496
50%,0.098,0.010,0.0,3.670,1.695,0.536
75%,0.203,0.020,0.0,3.871,1.974,0.565
max,1.009,0.152,0.0,5.272,3.483,0.676


# 6. Outlier detection (IQR rule)


In [21]:
def iqr_flags(s):
    q1, q3 = s.quantile([0.25, 0.75])
    k = 1.5 * (q3 - q1)
    return (s < q1 - k) | (s > q3 + k)

check_cols = ["dur_trim", "rms", "peak", "silence_ratio"]
flag_cols = [f"out_{c}" for c in check_cols]
for c in check_cols:
    df[f"out_{c}"] = iqr_flags(df[c])

print("Outliers per statistic:")
print(df[flag_cols].sum())
print("Clips flagged by at least one rule:", df[flag_cols].any(axis=1).sum())
print("Silent clips (rms < 1e-4):", (df["rms"] < 1e-4).sum())
print("Clips with clipping (>0.1% samples at full scale):", (df["clip_frac"] > 0.001).sum())

Outliers per statistic:
out_dur_trim          56
out_rms              148
out_peak             155
out_silence_ratio     80
dtype: int64
Clips flagged by at least one rule: 258
Silent clips (rms < 1e-4): 0
Clips with clipping (>0.1% samples at full scale): 0


Boxplots of each statistic by emotion. Points beyond the whiskers are the flagged outliers.

In [22]:
order = ["neutral", "calm", "happy", "sad", "angry", "fearful", "disgust", "surprised"]
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, c in zip(axes.ravel(), check_cols):
    sns.boxplot(data=df, x="emotion", y=c, order=order, ax=ax)
    ax.set_title(f"{c} by emotion")
    ax.tick_params(axis="x", rotation=40)
plt.tight_layout()
plt.savefig("../images/cleaning_outlier_boxplots.png", dpi=140)
plt.show()

**What the plot shows:** loudness (RMS, peak) clearly differs by emotion: angry, fearful and happy clips are louder, calm and sad are quieter. Duration and silence ratio also vary by emotion. Many IQR 'outliers' are therefore a natural result of how each emotion is spoken, not recording errors.

# 7. Inspect the flagged clips before deciding

In [23]:
flagged = df[df[flag_cols].any(axis=1)]
print("Flagged clips by emotion:")
print(flagged["emotion"].value_counts())
flagged[["filename", "emotion", "actor", "dur_trim", "rms", "peak", "silence_ratio"]].head(20)

Flagged clips by emotion:
emotion
angry        92
fearful      60
happy        29
sad          24
disgust      22
calm         21
surprised     9
neutral       1
Name: count, dtype: int64


,filename,emotion,actor,dur_trim,rms,peak,silence_ratio
32,03-01-05-02-01-01-01.wav,angry,1,2.020136,0.088748,0.813334,0.507779
33,03-01-05-02-01-02-01.wav,angry,1,2.252336,0.053624,0.551321,0.484717
34,03-01-05-02-02-01-01.wav,angry,1,2.020136,0.056003,0.585109,0.511745
35,03-01-05-02-02-02-01.wav,angry,1,2.345215,0.149243,0.989104,0.471534
38,03-01-06-01-02-01-01.wav,fearful,1,2.461315,0.008180,0.064026,0.283831
40,03-01-06-02-01-01-01.wav,fearful,1,2.786395,0.031053,0.327279,0.443282
41,03-01-06-02-01-02-01.wav,fearful,1,2.089796,0.040138,0.510605,0.506844
42,03-01-06-02-02-01-01.wav,fearful,1,1.787937,0.062700,0.579007,0.549713
43,03-01-06-02-02-02-01.wav,fearful,1,1.764717,0.042772,0.485229,0.547966
77,03-01-03-02-01-02-02.wav,happy,2,1.602177,0.032284,0.479506,0.563480


# 8. Apply decisions, cleaning report and saved files

Duplicate handling: if identical audio has the **same labels**, the later copy is dropped. If the labels **conflict**, nothing is dropped automatically and a warning is printed, because that needs a manual decision.

In [24]:
n_dup = int(df["md5"].duplicated().sum())
n_stereo = int((fmt["channels"] == 2).sum())
n_flagged = int(df[flag_cols].any(axis=1).sum())

# check whether duplicated audio carries conflicting labels
conflict = False
if n_dup > 0:
    conflict = bool((df[df["md5"].duplicated(keep=False)].groupby("md5")["emotion"].nunique() > 1).any())

if conflict:
    print("WARNING: identical audio with DIFFERENT emotion labels. Not dropping automatically - decide manually.")
    df_clean = df.copy()
    dup_action = "Label conflict - needs manual decision"
else:
    df_clean = df.drop_duplicates(subset="md5", keep="first").reset_index(drop=True)
    dup_action = f"Removed {n_dup} copy; {len(df_clean)} clips remain" if n_dup else "None needed"
print("Rows before:", len(df), "| after:", len(df_clean))

report = pd.DataFrame([
    ["Missing values (labels/metadata)", int(df[["emotion", "actor", "gender", "intensity"]].isna().sum().sum()), "None needed"],
    ["Total clips (expected 1440)", len(df), "None needed"],
    ["Clips per actor (expected 60)", str(df.groupby("actor").size().unique().tolist()), "None needed"],
    ["Unreadable/corrupt files", int((~fmt["readable"]).sum()), "None needed"],
    ["Sample rate inconsistent", int(fmt["sr"].nunique() > 1), "All files 48 kHz; resampled to 22.05 kHz"],
    ["Stereo files (others mono)", n_stereo, "Converted to mono on load"],
    ["Duplicate filenames", int(df["filename"].duplicated().sum()), "None needed"],
    ["Duplicate audio (MD5)", n_dup, dup_action],
    ["Silent clips", int((df["rms"] < 1e-4).sum()), "None needed"],
    ["Clipped clips", int((df["clip_frac"] > 0.001).sum()), "None needed"],
    ["IQR outliers (any rule)", n_flagged, "Kept: natural emotional variation"],
], columns=["Check", "Found", "Action"])

report.to_csv("../data/cleaning_report.csv", index=False)
df_clean.drop(columns=["md5"]).to_csv("../data/clean_metadata.csv", index=False)
report

Rows before: 1440 | after: 1439


,Check,Found,Action
0,Missing values (labels/metadata),0,None needed
1,Total clips (expected 1440),1440,None needed
2,Clips per actor (expected 60),[60],None needed
3,Unreadable/corrupt files,0,None needed
4,Sample rate inconsistent,0,All files 48 kHz; resampled to 22.05 kHz
5,Stereo files (others mono),5,Converted to mono on load
6,Duplicate filenames,0,None needed
7,Duplicate audio (MD5),1,Removed 1 copy; 1439 clips remain
8,Silent clips,0,None needed
9,Clipped clips,0,None needed


## 9. Summary

- The dataset is complete and consistent: no missing values, no unreadable files, one sample rate (48 kHz), one bit depth (16-bit).
- 5 stereo files are converted to mono. Duplicate audio is handled as described in section 8.
- Outliers are natural emotional variation and are kept.
- Class imbalance: neutral has half as many clips as the other emotions, so models use class weights.
- Saved: `data/cleaning_report.csv`, `data/clean_metadata.csv` (use this as the single source of truth for later notebooks), `images/cleaning_outlier_boxplots.png`.